# Demo 1 — An LLM is a next-word predictor
<a id="top"></a>

```yaml
title: Demo 1 — An LLM is a next-word predictor
keywords: next-word prediction, logits, probability distribution, greedy decoding, gpt2
estimated duration: 12
```

> **Lesson L01**, the first demo in the chain. Pairs with the written lecture
> [L0102_lecture.md](L0102_lecture.md) §1 and the roadmap
> [demos_or_activities.md](../../../../docs/origin/lesson_roadmaps/L01/demos_or_activities.md).
>
> You'll use a **local GPT-2** (small, 124M) here because you can read its raw next-token
> probabilities directly — offline, deterministic, no API key. The *mechanism* is identical
> for Claude; only the model is small enough to inspect.

**Contents**

1. [One forward pass is a distribution](#1-one-forward-pass-is-a-distribution)
2. [Generation is that loop, repeated](#2-generation-is-that-loop-repeated)
3. [The consequence: no memory](#3-the-consequence-no-memory)

In [1]:
from __future__ import annotations

import torch
import transformers
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

transformers.logging.set_verbosity_error()  # quiet advisory HF logs in committed output

# Load once. A small model is enough to SEE the mechanism: text in -> a distribution
# over the next token. First run downloads weights (~0.5 GB), cached afterwards.
TOKENIZER = GPT2TokenizerFast.from_pretrained("gpt2")
MODEL = GPT2LMHeadModel.from_pretrained("gpt2")
MODEL.eval()  # inference only; there is no training in this course


def next_token_distribution(prompt: str) -> torch.Tensor:
    """Probability distribution over the next token, given the text so far."""
    inputs = TOKENIZER(prompt, return_tensors="pt")
    with torch.no_grad():
        logits = MODEL(**inputs).logits[0, -1]  # scores for the position after `prompt`
    return torch.softmax(logits, dim=-1)


def top_next_tokens(prompt: str, k: int = 8) -> list[tuple[str, float]]:
    """The k most-likely next tokens as (decoded_piece, probability) pairs."""
    probs = next_token_distribution(prompt)
    top = torch.topk(probs, k)
    return [
        (TOKENIZER.decode([int(idx)]), float(p))
        for p, idx in zip(top.values, top.indices, strict=True)
    ]

## 1. One forward pass is a distribution

Give the model some text; it hands back a **probability for every token in its vocabulary**.
It does not *know* the answer — it *ranks* candidates. Watch it hedge.

In [2]:
PROMPT = "Water is made of hydrogen and"
for piece, prob in top_next_tokens(PROMPT):
    print(f"{prob:6.3f}  {piece!r}")

 0.464  ' oxygen'
 0.279  ' helium'
 0.034  ' carbon'
 0.019  ' methane'
 0.010  ' is'
 0.010  ' sulfur'
 0.009  ' chlorine'
 0.008  ' nitrogen'


` oxygen` leads — but notice the model spreads real probability across ` helium`, ` carbon`,
` nitrogen`, … It ranked oxygen highest; it did not "know" it. The word to hold onto is
**distribution**: model scale (Demo 3.5) and temperature (Demo 3.6) both *reshape this exact
distribution*.

[↑ Back to top](#top)

## 2. Generation is that loop, repeated

"Writing a sentence" is not a special mode. It is: predict the next token, append it, and
predict again. Below you always take the single most-likely token (**greedy decoding**) and
watch a sentence assemble one token at a time.

In [3]:
def greedy_continue(prompt: str, n_tokens: int = 6) -> str:
    """Append the most-likely token `n_tokens` times, printing the text as it grows."""
    text = prompt
    for _ in range(n_tokens):
        probs = next_token_distribution(text)
        best_id = int(torch.argmax(probs))
        text += TOKENIZER.decode([best_id])
        print(text)
    return text


_ = greedy_continue(PROMPT, n_tokens=6)

Water is made of hydrogen and oxygen
Water is made of hydrogen and oxygen.
Water is made of hydrogen and oxygen. The
Water is made of hydrogen and oxygen. The hydrogen
Water is made of hydrogen and oxygen. The hydrogen is
Water is made of hydrogen and oxygen. The hydrogen is made


## 3. The consequence: no memory

Each forward pass sees **only the text passed in** — nothing else. The model kept no state
between the calls above; `greedy_continue` fed the whole growing string back in every step.

That is the load-bearing fact of the whole lesson: a chat *feels* continuous only because your
client re-sends the entire history every call. That re-sent text is re-read — and later
(Demo 5) re-billed — every single time. Everything ahead (front-loading, preambles, the
window, cost) is a consequence of this one fact.

[↑ Back to top](#top)